## Logistic Regression
### Predict exam pass/fail based on a collected dataset

Thực hành phân loại nhị phân với dữ liệu điểm thi và email.


In [1]:
from pathlib import Path

project_dir = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data" / "final_grades.xlsx").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Run this notebook from the project or a lab directory.")
data_dir = project_dir / "data"

# Load data
import pandas as pd

data_path = data_dir / "student_performance_prediction.csv"
exam_df = pd.read_csv(data_path)

In [2]:
exam_df.head()

### Exploratory Data Analysis

In [3]:
# Show descriptive statistics
exam_df.describe()
print(len(exam_df))

40000


In [4]:
# Check missing values
print(exam_df.isna().sum())

Student ID                                        0
Study Hours per Week                           1995
Attendance Rate                                1992
Previous Grades                                1994
Participation in Extracurricular Activities    2000
Parent Education Level                         2000
Passed                                         2000
dtype: int64


In [5]:
# Remove missing values

exam_df = exam_df.dropna(subset=["Study Hours per Week", "Passed"]).copy()

In [6]:
# Check again

exam_df.describe()

In [7]:
print(exam_df.columns)

Index(['Student ID', 'Study Hours per Week', 'Attendance Rate',
       'Previous Grades', 'Participation in Extracurricular Activities',
       'Parent Education Level', 'Passed'],
      dtype='str')


In [8]:
# Get study hour per week as single input (X), Passed as output (y)

# Encoding passed/fail labels
exam_df['Label'] = exam_df.apply(lambda x:1 if x['Passed'] == 'Yes' else 0, axis=1)

In [9]:
exam_df.head()

In [10]:
import numpy as np

X = np.asarray(exam_df['Study Hours per Week'])
y = np.asarray(exam_df['Label'])

In [11]:
print(X.shape, y.shape)

(36095,) (36095,)


In [12]:
X = X.reshape(-1,1)
# y = y.reshape(-1,1)

In [13]:
# Split the dataset into train set and test set

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, 
                                                    test_size=0.2, 
                                                   stratify=y, random_state=42)

In [14]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(28876, 1) (28876,)
(7219, 1) (7219,)


In [15]:
y_test.shape

In [16]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
 
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=1000))
clf.fit(X_train, y_train)
print(f"Logistic Regression test accuracy: {clf.score(X_test, y_test):.2%}")
majority_class = np.bincount(y_train).argmax()
print(f"Majority-class baseline: {np.mean(y_test == majority_class):.2%}")
 
proba = clf.predict_proba(X_test[:5])[:, 1]
print(proba.round(3))      # [0.999 0.004 1.    0.    0.035]
print(y_test[:5])             # [1 0 1 0 0]

Logistic Regression test accuracy: 49.98%
Majority-class baseline: 50.06%
[0.501 0.504 0.5   0.504 0.505]
[1 1 0 0 0]


## Naive Bayes Classification

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
 
emails = ["win free money now", "free prize claim now",
          "cheap money offer win", "meeting schedule for monday",
          "lunch with the team", "project report for monday"]
labels = [1, 1, 1, 0, 0, 0]                    # 1 = spam
vec = CountVectorizer()
nb = MultinomialNB(alpha=1).fit(vec.fit_transform(emails), labels)
 
for s in ["free money", "team meeting on monday",
          "free lunch with the team"]:
    print(s, nb.predict_proba(vec.transform([s]))[0, 1].round(3))
# free money 0.9
# team meeting on monday 0.077
# free lunch with the team 0.158   <- one spammy word is not enough


free money 0.9
team meeting on monday 0.077
free lunch with the team 0.158


### Real example of spam email classification using Multinomial Naive Bayes

In [18]:
# Load the dataset

email_data_path = data_dir / "spam_or_not_spam.csv"
email_df = pd.read_csv(email_data_path)

In [19]:
email_df.head()

In [20]:
email_df.describe()

In [21]:
print(email_df.isna().sum())

email    1
label    0
dtype: int64


In [22]:
email_df = email_df.dropna(subset=["email", "label"]).copy()
print(email_df.isna().sum())

email    0
label    0
dtype: int64


In [23]:
email_df.describe()

In [24]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Get email content from the dataset
emails = email_df['email']
labels = email_df['label']

In [25]:
# Init text feature extraction
vec = CountVectorizer()

# Fit the model from the data
nb = MultinomialNB(alpha=1).fit(vec.fit_transform(emails), labels)

In [26]:
# Make some prediction

for s in ["free money", "team meeting on monday",
          "free lunch with the team"]:
    print(s, nb.predict_proba(vec.transform([s]))[0, 1].round(3))

free money 0.883
team meeting on monday 0.003
free lunch with the team 0.409


## Model Evaluation

In [27]:
# Split the email dataset into train/test sets
from sklearn.model_selection import train_test_split

email_train, email_test, label_train, label_test = \
train_test_split(emails, labels, test_size=0.2, stratify=labels, random_state=42)

# Train the model
vec = CountVectorizer()

# Fit the model from the data
nb = MultinomialNB(alpha=1).fit(vec.fit_transform(email_train), label_train)

In [28]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

predicted_labels = nb.predict(vec.transform(email_test))
print(f"Test accuracy: {accuracy_score(label_test, predicted_labels):.2%}")
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(label_test, predicted_labels, labels=[0, 1]))
print(classification_report(
    label_test, predicted_labels, labels=[0, 1],
    target_names=["Not spam", "Spam"], zero_division=0,
))

Test accuracy: 98.83%
Confusion matrix (rows = actual, columns = predicted):
[[500   0]
 [  7  93]]
              precision    recall  f1-score   support

    Not spam       0.99      1.00      0.99       500
        Spam       1.00      0.93      0.96       100

    accuracy                           0.99       600
   macro avg       0.99      0.97      0.98       600
weighted avg       0.99      0.99      0.99       600

